# 🧬 Biohub Cell Tracking — Classical Baseline (with EDA)

**Task.** Each sample is a **3D + time light-sheet microscopy movie** of a developing *zebrafish*
embryo — a `(T, Z, Y, X)` fluorescence volume (Chan Zuckerberg Biohub / **Zebrahub**). For every movie
we reconstruct a **cell-lineage tracking graph**:

| element | meaning | CSV row |
|---|---|---|
| **node** | one detected cell at one timepoint `(t, z, y, x)` | `row_type = node` |
| **edge** | a temporal link `source → target` (same cell next frame, or a division) | `row_type = edge` |

So the pipeline is **detect → link**: find cell centres in each 3-D frame, then connect each cell to
its next-frame self. Score = graph-matching metric in `[0,1]` (point matching ≈7 µm + edge/lineage
accuracy).

This is a compact, **no-GPU / no-internet** classical baseline (**public LB ≈ 0.75**) with an **EDA
section** so the approach is easy to follow. One-line pipeline:

> stream Zarr frame → XY-pool ×4 (≈isotropic) → smooth → robust threshold → **3D local-maxima**
> (moderate density) → intensity-weighted centroid → physical NMS → **two-pass µm-gated Hungarian
> linking** → prune isolated nodes → `submission.csv`.

Read the **§EDA** below (data, images, ground truth, and the key tuning lesson), then the pipeline.


## 1 · Config & paths

In [1]:

from __future__ import annotations
import os, sys, json, glob, time, gc
from dataclasses import dataclass, field
from collections import defaultdict
from pathlib import Path
import numpy as np, pandas as pd
from scipy.ndimage import gaussian_filter, maximum_filter
from scipy.optimize import linear_sum_assignment
from scipy.spatial import cKDTree
try:
    from skimage.feature import peak_local_max
    from skimage.filters import threshold_otsu
    _SK = True
except Exception:
    _SK = False

SCALE_ZYX = np.array([1.625, 0.40625, 0.40625])      # (Z,Y,X) µm/voxel — physical scale

# resolve competition root: Kaggle first, then local ./data
CANDIDATES = [
    "/kaggle/input/biohub-cell-tracking-during-development",
    "/kaggle/input/competitions/biohub-cell-tracking-during-development",
    "data",
]
ROOT = next((p for p in CANDIDATES if Path(p, "test").exists()), "data")
TEST_DIR  = Path(ROOT) / "test"
TRAIN_DIR = Path(ROOT) / "train"
OUT = "submission.csv"   # Kaggle notebooks run with CWD=/kaggle/working, so this lands in the right place
print("data root:", ROOT, "| test movies:", len(list(TEST_DIR.glob('*.zarr'))))

@dataclass
class Config:
    scale: np.ndarray = field(default_factory=lambda: SCALE_ZYX.copy())
    # detection (single-scale local-maxima — MODERATE count; this density scored 0.720 on the LB.
    #            LB lesson: over-detecting hurts the metric, so keep the count moderate.)
    xy_ds: int = 4
    smooth_sigma: float = 1.0
    thresh_rel: float = 0.28
    use_otsu: bool = True
    min_peak_dist: int = 2
    nms_radius_um: float = 3.5
    border_z: int = 0                # keep z-edge peaks — GT cells live at z=1 / z=Z-1
    max_nodes_per_frame: int = 420
    refine_rz: int = 2
    refine_ryx: int = 5
    # linking (v4: two-pass — tight gate first, then full gate for leftovers)
    max_link_dist_um: float = 10.0   # full gate (≈ p99 of GT displacement)
    tight_gate_um: float = 6.0       # pass-1 tight raw gate → kills distractor steals
    use_motion: bool = True
    # division — OFF by default: GT mitoses are rare; FP divisions hurt the division score
    detect_divisions: bool = False
    div_parent_dist_um: float = 9.0
    div_sister_dist_um: float = 9.0
    div_min_count_gain: int = 1
    # pruning
    prune_isolated: bool = True
    keep_strong_isolated_q: float = 0.0

CFG = Config()


## 2 · Readers — Zarr image (stream one frame) + GEFF ground-truth

In [2]:

def read_array_meta(zarr_path):
    with open(Path(zarr_path) / "0" / "zarr.json") as f:
        m = json.load(f)
    return dict(shape=tuple(m["shape"]), dtype=np.dtype(m["data_type"]),
                chunks=tuple(m["chunk_grid"]["configuration"]["chunk_shape"]))

_ZC = {}
def load_volume(zarr_path, t, meta=None):
    "One timepoint (Z,Y,X). Prefer the zarr lib; fall back to raw blosc2 chunk."
    try:
        import zarr
        key = str(zarr_path)
        if key not in _ZC: _ZC[key] = zarr.open(key, mode="r")["0"]
        return np.asarray(_ZC[key][t])
    except Exception:
        import blosc2
        if meta is None: meta = read_array_meta(zarr_path)
        chunk = Path(zarr_path) / "0" / "c" / str(t) / "0" / "0" / "0"
        buf = blosc2.decompress(open(chunk, "rb").read())
        return np.frombuffer(buf, dtype=meta["dtype"]).reshape(meta["shape"][1:])

def read_geff(geff_path):
    "GT graph -> nodes_df[node_id,t,z,y,x] + edges_df[source_id,target_id]; local validation only."
    import networkx as nx
    from geff import read as geff_read
    obj = geff_read(str(geff_path), backend="networkx")
    g = obj[0] if isinstance(obj, tuple) else obj
    nodes = pd.DataFrame([{"node_id": int(i), "t": int(round(float(d["t"]))),
                           "z": float(d["z"]), "y": float(d["y"]), "x": float(d["x"])}
                          for i, d in g.nodes(data=True)]).sort_values(["t","node_id"]).reset_index(drop=True)
    edges = pd.DataFrame([{"source_id": int(u), "target_id": int(v)} for u, v in g.edges()],
                         columns=["source_id","target_id"])
    return nodes, edges


<!--EDA-CELL-->
## 📊 EDA — understand the data before the model
*(Read-only exploration; these cells never affect `submission.csv`, and each is wrapped so a missing
optional dependency can't break the run.)*

In [ ]:
# EDA-CELL

# dataset inventory + geometry (needs only JSON metadata → always works)
try:
    import glob, numpy as np, pandas as pd
    from pathlib import Path
    eda_rows = []
    for eda_split in ["train", "test"]:
        for eda_zp in sorted((Path(ROOT)/eda_split).glob("*.zarr")):
            try:
                em = read_array_meta(eda_zp); T,Z,Y,X = em["shape"]
                eda_rows.append(dict(split=eda_split, embryo=eda_zp.name[:4], dataset=eda_zp.name[:-5],
                                     T=T, Z=Z, Y=Y, X=X, dtype=str(em["dtype"])))
            except Exception:
                pass
    eda_inv = pd.DataFrame(eda_rows)
    if len(eda_inv):
        print("Movies per split × embryo (two embryos: 44b6, 6bba):")
        display(eda_inv.pivot_table(index="embryo", columns="split", values="dataset", aggfunc="count", fill_value=0))
        print("\\nExample geometry (voxels):"); display(eda_inv.head(3))
        print(f"voxel size (Z,Y,X) = {tuple(SCALE_ZYX)} µm  →  Z is ~{SCALE_ZYX[0]/SCALE_ZYX[2]:.0f}× coarser than X/Y")
    else:
        print("no .zarr movies found under", ROOT)
except Exception as e:
    print("inventory EDA skipped:", repr(e))


<!--EDA-CELL-->
### The images — bright nuclei on a dark, anisotropic 3-D grid
Maximum-intensity projections (MIP) of one frame. The **Z-MIP** (top view) shows well-separated
nuclei; the **Y-MIP** (side view) shows the strong **Z-anisotropy** (voxels ~4× taller in Z).

In [ ]:
# EDA-CELL

# image MIPs + intensity histogram (needs only the image reader → works on Kaggle via blosc2)
try:
    import numpy as np, matplotlib.pyplot as plt
    from pathlib import Path
    eda_img = None
    for eda_zp in sorted(Path(TEST_DIR).glob("*.zarr")):
        if (eda_zp/"0"/"zarr.json").exists() and (eda_zp/"0"/"c"/"0"/"0"/"0"/"0").exists():
            eda_img = eda_zp; break
    if eda_img is not None:
        em = read_array_meta(eda_img); eda_tt = em["shape"][0]//2
        eda_vol = load_volume(eda_img, eda_tt, em)
        fig, ax = plt.subplots(1, 3, figsize=(14, 4.2))
        ax[0].imshow(eda_vol.max(0), cmap="magma"); ax[0].set_title(f"{eda_img.name[:-5]}  Z-MIP  (t={eda_tt})")
        ax[0].set_xlabel("x"); ax[0].set_ylabel("y")
        ax[1].imshow(eda_vol.max(1), cmap="magma", aspect=float(SCALE_ZYX[0]/SCALE_ZYX[2]))
        ax[1].set_title("Y-MIP (Z×X) — note Z anisotropy"); ax[1].set_xlabel("x"); ax[1].set_ylabel("z")
        ax[2].hist(eda_vol[eda_vol > eda_vol.min()].ravel(), bins=80, color="#2563eb")
        ax[2].set_yscale("log"); ax[2].set_title("voxel intensity (log count)"); ax[2].set_xlabel("intensity")
        plt.tight_layout(); plt.show()
        print(f"volume {eda_vol.shape} {eda_vol.dtype}; intensity {int(eda_vol.min())}..{int(eda_vol.max())}, median {int(np.median(eda_vol))}")
    else:
        print("no local test image frames available for the MIP panel")
except Exception as e:
    print("image EDA skipped:", repr(e))


<!--EDA-CELL-->
### The ground truth is a **sparse graph** (GEFF)
Each train movie ships a `.geff` graph — **nodes** `(t,z,y,x)` + **edges** `source→target`. The single
most important fact: the GT is **sparse** (only a few tracked cells per frame) and **divisions are
rare**, so **over-detecting is penalised**. Summary measured offline from the training GEFF graphs:

In [ ]:
# EDA-CELL

# GT sparsity — static summary (Kaggle has no zarr/geff, so we show offline-measured numbers)
try:
    import pandas as pd, matplotlib.pyplot as plt
    eda_gt = pd.DataFrame([
        ["44b6", 73, 236, 2.5, "~0.2"],
        ["6bba", 22, 1045, 10.4, "~1.5"],
    ], columns=["embryo", "movies", "mean_GT_nodes/movie", "GT_cells/frame", "divisions/movie"])
    display(eda_gt)
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
    ax[0].bar(eda_gt.embryo, eda_gt["GT_cells/frame"], color=["#2563eb", "#dc2626"])
    ax[0].set_title("GT tracked cells per frame (SPARSE)"); ax[0].set_ylabel("cells / frame")
    ax[1].bar(["images/frame\\n(hundreds)", "GT-tracked\\n(few)"], [300, 8], color=["#94a3b8", "#16a34a"])
    ax[1].set_title("why over-detection hurts"); ax[1].set_ylabel("count (illustrative)")
    plt.tight_layout(); plt.show()
    print("→ Only a handful of cells/frame are annotated. Detect a MODERATE, accurate set + link well.")
except Exception as e:
    print("GT EDA skipped:", repr(e))


<!--EDA-CELL-->
### The evaluation metric & the key lesson from tuning
The score blends **detection** (matched nodes ≈7 µm), **linking** (correct temporal edges) and
**divisions**, with a **node-count adjustment**. Because the GT is sparse, the metric **punishes
over-detection**. Our own leaderboard experiments make this concrete:

In [ ]:
# EDA-CELL

# what we learned on the public LB (this explains every design choice)
try:
    import pandas as pd
    eda_lb = pd.DataFrame([
        ["moderate density (~250/frame)",    "Hungarian (global)",   0.720],
        ["OVER-detect DoG (~300-700/frame)", "Hungarian (global)",   0.593],
        ["OVER-detect DoG (~300-700/frame)", "two-pass tight-gate",  0.662],
        ["moderate density (~250/frame)",    "two-pass tight-gate",  0.750],
    ], columns=["detection", "linking", "public_LB"])
    display(eda_lb)
    print("Lesson 1 — MORE detections HURT: over-detection triggers the node-count penalty AND adds")
    print("          distractors that corrupt linking (0.72 → 0.59).")
    print("Lesson 2 — a TIGHT (~6 µm) two-pass link gate fixes distractor 'steals' (0.72 → 0.75),")
    print("          because real cells move <=7 µm/frame and neighbours are >=12 µm apart.")
except Exception as e:
    print(e)


## 3 · Detection — 3D local-maxima at a *moderate* density

Detect bright nuclei as 3-D local maxima on the XY-pooled (≈isotropic) volume, above a robust
(Otsu ∨ relative-rise) threshold, then refine each to an intensity-weighted centroid and de-duplicate
with a physical-distance NMS. **Key calibration:** a *moderate* number of detections per frame
(~200–300) scores far better than flooding the frame — see the EDA lesson table above (over-detection
dropped the LB from 0.72 → 0.59).

In [3]:

def _block_mean_xy(vol, f):
    Z, Y, X = vol.shape; Y2, X2 = (Y//f)*f, (X//f)*f
    v = vol[:, :Y2, :X2].astype(np.float32, copy=False)
    return v.reshape(Z, Y2//f, f, X2//f, f).mean(axis=(2, 4))

def _robust_threshold(sm, cfg):
    bg = float(np.median(sm)); hi = float(np.percentile(sm, 99.9))
    rel = bg + cfg.thresh_rel * max(hi - bg, 1e-6)
    if cfg.use_otsu and _SK:
        try: return max(float(threshold_otsu(sm)), rel)
        except Exception: pass
    return max(float(np.percentile(sm, 96.0)), rel)

def _peaks(sm, thr, d):
    if _SK:
        return peak_local_max(sm, min_distance=int(d), threshold_abs=thr, exclude_border=False).astype(np.int32)
    size = 2*int(d)+1; mx = maximum_filter(sm, size=size, mode="nearest")
    return np.argwhere((sm >= mx) & (sm > thr)).astype(np.int32)

def _refine(vol, zyx, cfg):
    Z, Y, X = vol.shape; z, y, x = (int(round(v)) for v in zyx)
    z0,z1 = max(0,z-cfg.refine_rz), min(Z,z+cfg.refine_rz+1)
    y0,y1 = max(0,y-cfg.refine_ryx), min(Y,y+cfg.refine_ryx+1)
    x0,x1 = max(0,x-cfg.refine_ryx), min(X,x+cfg.refine_ryx+1)
    crop = vol[z0:z1, y0:y1, x0:x1].astype(np.float32); bg = float(crop.min())
    w = np.clip(crop-bg, 0, None); s = float(w.sum())
    if s <= 0: return np.array([z,y,x], float), 0.0
    zz,yy,xx = np.mgrid[z0:z1, y0:y1, x0:x1]
    return np.array([(zz*w).sum(),(yy*w).sum(),(xx*w).sum()])/s, float(crop.max()-bg)

def _physical_nms(coords, scores, radius_um, scale):
    if len(coords) <= 1: return coords, scores
    pts = coords*scale[None,:]; order = np.argsort(-scores)
    tree = cKDTree(pts); killed = np.zeros(len(coords), bool); keep = []
    for i in order:
        if killed[i]: continue
        keep.append(int(i)); killed[tree.query_ball_point(pts[i], r=radius_um)] = True
    keep = np.array(keep); return coords[keep], scores[keep]

def detect(vol, cfg):
    # single-scale local maxima on a robustly thresholded, XY-pooled grid (moderate detection count)
    f = cfg.xy_ds
    pooled = _block_mean_xy(vol, f) if f > 1 else vol.astype(np.float32)
    sm = gaussian_filter(pooled, cfg.smooth_sigma) if cfg.smooth_sigma > 0 else pooled
    pk = _peaks(sm, _robust_threshold(sm, cfg), cfg.min_peak_dist)
    if len(pk) == 0: return np.zeros((0,3)), np.zeros(0)
    full = pk.astype(float)
    full[:,1] = full[:,1]*f + (f-1)/2; full[:,2] = full[:,2]*f + (f-1)/2
    Z = vol.shape[0]; coords, scores = [], []
    for p in full:
        if p[0] < cfg.border_z or p[0] > Z-1-cfg.border_z: continue
        c, s = _refine(vol, p, cfg); coords.append(c); scores.append(s)
    if not coords: return np.zeros((0,3)), np.zeros(0)
    coords = np.array(coords); scores = np.array(scores)
    coords, scores = _physical_nms(coords, scores, cfg.nms_radius_um, cfg.scale)
    if cfg.max_nodes_per_frame and len(coords) > cfg.max_nodes_per_frame:
        k = np.argsort(-scores)[:cfg.max_nodes_per_frame]; coords, scores = coords[k], scores[k]
    return coords, scores


## 4 · Linking (Hungarian, µm-gated, motion-aware) + conservative divisions

In [4]:

def _link(prev_xyz, curr_xyz, cfg, prev_vel=None):
    # v4: TWO-PASS Hungarian. Pass 1 uses a TIGHT raw-displacement gate (~6µm) on
    # motion-predicted positions — since cells move <=7µm (p99) and neighbours are >=12µm
    # apart, this kills the distractor "steals" that a single global pass commits to.
    # Pass 2 re-links the few leftovers at the full gate (rare fast cells).
    if len(prev_xyz)==0 or len(curr_xyz)==0: return []
    P = prev_xyz*cfg.scale[None,:]; C = curr_xyz*cfg.scale[None,:]
    pred = P + (0.5*prev_vel if (cfg.use_motion and prev_vel is not None) else 0.0)
    N, M = len(P), len(C); BIG = 1e9
    if N*M > 4_000_000:                                   # greedy fallback for pathological frames
        tree = cKDTree(C); cand = []
        for i,p in enumerate(P):
            for j in tree.query_ball_point(p, r=cfg.max_link_dist_um):
                cand.append((float(np.linalg.norm(p-C[j])), i, int(j)))
        cand.sort(); up,uc,out = set(),set(),[]
        for d,i,j in cand:
            if i in up or j in uc: continue
            up.add(i); uc.add(j); out.append((i,j))
        return out
    def _hun(pi, ci, gate):
        if len(pi)==0 or len(ci)==0: return []
        Draw = np.sqrt(((P[pi][:,None]-C[ci][None])**2).sum(2))
        D = np.sqrt(((pred[pi][:,None]-C[ci][None])**2).sum(2))
        cost = np.where(Draw > gate, BIG, D)
        ri, rc = linear_sum_assignment(cost)
        return [(int(pi[r]),int(ci[c])) for r,c in zip(ri,rc) if cost[r,c] < BIG]
    tight = min(cfg.tight_gate_um, cfg.max_link_dist_um)
    links = _hun(np.arange(N), np.arange(M), tight)
    up = {p for p,_ in links}; uc = {c for _,c in links}
    fp = np.array([i for i in range(N) if i not in up], int)
    fc = np.array([j for j in range(M) if j not in uc], int)
    links += _hun(fp, fc, cfg.max_link_dist_um)
    return links

def _divisions(prev_xyz, curr_xyz, links, cfg):
    if not cfg.detect_divisions or len(curr_xyz)-len(prev_xyz) < cfg.div_min_count_gain: return []
    P = prev_xyz*cfg.scale[None,:]; C = curr_xyz*cfg.scale[None,:]
    matched = {c for _,c in links}; parent_of = {c:p for p,c in links}
    free = [j for j in range(len(curr_xyz)) if j not in matched]
    if not free: return []
    ptree = cKDTree(P); extra = []
    for j in free:
        d, p = ptree.query(C[j], k=1)
        if d > cfg.div_parent_dist_um: continue
        sisters = [c for c,pp in parent_of.items() if pp == p]
        if not sisters: continue
        sis = min(sisters, key=lambda c: np.linalg.norm(C[c]-C[j]))
        if np.linalg.norm(C[sis]-C[j]) <= cfg.div_sister_dist_um: extra.append((int(p),int(j)))
    return extra


## 5 · Track a whole movie → node/edge rows

In [5]:

COLS = ["dataset","row_type","node_id","t","z","y","x","source_id","target_id"]

def track_movie(load_vol, T, dataset, cfg, t_max=0, verbose=False):
    node_rows, edge_rows, node_score = [], [], {}
    prev_ids, prev_xyz, prev_vel = [], np.zeros((0,3)), None
    nid, counts, ndiv = 1, [], 0
    if t_max: T = min(T, t_max)
    for t in range(T):
        vol = load_vol(t); coords, scores = detect(vol, cfg); del vol; gc.collect()
        ids = list(range(nid, nid+len(coords))); nid += len(coords)
        for i,c,s in zip(ids, coords, scores):
            node_rows.append((dataset,"node",i,t,float(c[0]),float(c[1]),float(c[2]),-1,-1)); node_score[i]=float(s)
        if t>0 and len(prev_ids):
            links = _link(prev_xyz, coords, cfg, prev_vel)
            extra = _divisions(prev_xyz, coords, links, cfg)
            vel = np.zeros((len(prev_xyz),3))
            for p,c in links:
                edge_rows.append((dataset,"edge",-1,-1,-1,-1,-1,prev_ids[p],ids[c]))
                vel[p] = (coords[c]-prev_xyz[p])*cfg.scale
            for p,c in extra: edge_rows.append((dataset,"edge",-1,-1,-1,-1,-1,prev_ids[p],ids[c]))
            ndiv += len({p for p,_ in extra})
            nv = np.zeros((len(coords),3))
            for p,c in links: nv[c] = vel[p]
            prev_vel = nv
        else: prev_vel = None
        prev_ids, prev_xyz = ids, coords; counts.append(len(coords))
        if verbose and ((t+1)%20==0 or t==T-1): print(f"  [{dataset}] t={t+1}/{T} cells={len(coords)}")
    nodes = pd.DataFrame(node_rows, columns=COLS); edges = pd.DataFrame(edge_rows, columns=COLS)
    if cfg.prune_isolated and len(edges):
        used = set(edges.source_id) | set(edges.target_id)
        if cfg.keep_strong_isolated_q > 0 and node_score:
            fl = np.quantile(list(node_score.values()), cfg.keep_strong_isolated_q)
            used |= {i for i,s in node_score.items() if s >= fl}
        nodes = nodes[nodes.node_id.isin(used)].reset_index(drop=True)
    return nodes, edges, dict(dataset=dataset, n_nodes=len(nodes), n_edges=len(edges),
                              cells_per_frame=float(np.mean(counts)) if counts else 0, n_div=ndiv, T=T)


## 6 · Local validation (grouped by embryo)

We score predictions against the train GEFF graphs. `proxy_score` mirrors the host logic
(per-frame point matching within 7 µm → node F1 + edge-Jaccard + division F1); `traccuracy_eval`
runs the official metric panel (CHOTA / Node F1 / Edge F1 / Division). Absolute numbers are a guide —
use them to **rank** settings, and always check **both embryos**.

In [6]:

def _split(df):
    n = df[df.row_type=="node"][["node_id","t","z","y","x"]].copy()
    e = df[df.row_type=="edge"][["source_id","target_id"]].astype(int).copy()
    return n, e

def match_nodes(pn, gn, r=7.0, scale=SCALE_ZYX):
    p2g, BIG = {}, 1e6
    for t in sorted(set(pn.t) & set(gn.t)):
        p = pn[pn.t==t].reset_index(drop=True); g = gn[gn.t==t].reset_index(drop=True)
        if len(p)==0 or len(g)==0: continue
        P = p[["z","y","x"]].values*scale; G = g[["z","y","x"]].values*scale
        D = np.sqrt(((P[:,None]-G[None])**2).sum(2)); cost = np.where(D<=r, D, BIG)
        ri, ci = linear_sum_assignment(cost)
        for a,b in zip(ri,ci):
            if cost[a,b] < BIG: p2g[int(p.loc[a,"node_id"])] = int(g.loc[b,"node_id"])
    return p2g

def _divset(e):
    if len(e)==0: return set()
    o = e.groupby("source_id").size(); return set(o[o>=2].index)

def proxy_score(pred_df, gn, ge, r=7.0, w=(0.5,0.4,0.1)):
    pn, pe = _split(pred_df); p2g = match_nodes(pn, gn, r)
    tp = len(p2g); fp = len(pn)-tp; fn = len(gn)-tp
    dp = tp/max(tp+fp,1); dr = tp/max(tp+fn,1); df1 = 2*dp*dr/max(dp+dr,1e-9)
    gset = set(map(tuple, ge[["source_id","target_id"]].astype(int).values))
    pm = {(p2g[s],p2g[t]) for s,t in pe.values if s in p2g and t in p2g}
    etp = len(pm & gset); ep = etp/max(len(pm),1); er = etp/max(len(gset),1)
    ef1 = 2*ep*er/max(ep+er,1e-9)
    gd = _divset(ge); pd_ = {p2g[s] for s in _divset(pe) if s in p2g}
    dtp = len(pd_ & gd); dpp = dtp/max(len(pd_),1); dpr = dtp/max(len(gd),1)
    divf1 = 2*dpp*dpr/max(dpp+dpr,1e-9) if (gd or pd_) else 1.0
    score = w[0]*df1 + w[1]*ef1 + w[2]*divf1
    return round(score,4), dict(det_recall=round(dr,3), det_precision=round(dp,3), det_f1=round(df1,3),
                                link_recall=round(er,3), link_precision=round(ep,3), link_f1=round(ef1,3),
                                div_f1=round(divf1,3), pred_nodes=len(pn), gt_nodes=len(gn),
                                node_tp=tp, node_fp=fp, node_fn=fn, edge_tp=etp, pred_edges=len(pe), gt_edges=len(gset))

def traccuracy_eval(pred_df, gn, ge, r=7.0, scale=SCALE_ZYX):
    import networkx as nx
    from traccuracy import TrackingGraph, run_metrics
    from traccuracy.matchers import PointMatcher
    from traccuracy.metrics import BasicMetrics, TrackOverlapMetrics, DivisionMetrics, CHOTAMetric
    def tg(n,e):
        g = nx.DiGraph()
        for r_ in n.itertuples(index=False): g.add_node(int(r_.node_id), t=int(r_.t), z=float(r_.z), y=float(r_.y), x=float(r_.x))
        for s,t in e[["source_id","target_id"]].astype(int).values:
            if g.has_node(s) and g.has_node(t): g.add_edge(int(s),int(t))
        return TrackingGraph(g, frame_key="t", location_keys=("z","y","x"))
    pn, pe = _split(pred_df)
    res,_ = run_metrics(tg(gn,ge), tg(pn,pe), PointMatcher(threshold=r, scale_factor=tuple(scale)),
                        [BasicMetrics(), TrackOverlapMetrics(), DivisionMetrics(), CHOTAMetric()])
    flat={}
    def walk(o,p=""):
        if isinstance(o,dict):
            for k,v in o.items(): walk(v,f"{p}{k}.")
        elif isinstance(o,(list,tuple)):
            for v in o: walk(v,p)
        else: flat[p.rstrip(".")]=o
    walk(res); return flat


In [7]:

def avail_T(zp):
    "Frames actually present on disk (full on Kaggle; possibly partial locally)."
    meta = read_array_meta(zp); T = meta["shape"][0]
    present = [t for t in range(T) if (Path(zp)/"0"/"c"/str(t)/"0"/"0"/"0").exists()]
    return (max(present)+1 if present else 0), meta

# LOCAL VALIDATION — optional. Needs `geff` (GT graphs) which is ABSENT on an offline Kaggle
# Code-Competition run; the whole block auto-skips there so it never blocks the submission below.
val = None
try:
    import geff  # noqa: F401  (only available where the GT GEFF graphs are)
    def pick(emb):
        for g in sorted(TRAIN_DIR.glob(f"{emb}_*.geff")):
            ds = g.name.replace(".geff","")
            for base in (TEST_DIR, TRAIN_DIR):
                zp = base/f"{ds}.zarr"
                if (zp/"0"/"zarr.json").exists():
                    T, _ = avail_T(zp)
                    if T > 1: return ds, zp, T
        return None
    val_targets = [p for p in (pick("6bba"), pick("44b6")) if p]
    rows = []
    for ds, zp, T in val_targets:
        meta = read_array_meta(zp)
        nodes, edges, st = track_movie(lambda t: load_volume(zp,t,meta), T, ds, CFG, verbose=True)
        gn, ge = read_geff(TRAIN_DIR/f"{ds}.geff")
        gn = gn[gn.t < T]; ge = ge[ge.source_id.isin(gn.node_id) & ge.target_id.isin(gn.node_id)]
        sub = pd.concat([nodes,edges], ignore_index=True)
        sc, br = proxy_score(sub, gn, ge)
        chota = nf1 = ef1 = np.nan
        try:
            fl = traccuracy_eval(sub, gn, ge)
            chota = round(float(fl.get("results.CHOTA", np.nan)),3)
            nf1 = round(float(fl.get("results.Node F1", np.nan)),3); ef1 = round(float(fl.get("results.Edge F1", np.nan)),3)
        except Exception as ex:
            print("  traccuracy panel skipped for", ds, "->", str(ex)[:60])
        rows.append(dict(dataset=ds, embryo=ds[:4], frames=T,
                         node_recall=br["det_recall"], link_recall=br["link_recall"], link_f1=br["link_f1"],
                         CHOTA=chota, NodeF1=nf1, EdgeF1=ef1, pred_nodes=br["pred_nodes"], gt_nodes=br["gt_nodes"]))
        print(f"  {ds}: node_recall={br['det_recall']:.3f}  link_recall={br['link_recall']:.3f}  CHOTA={chota}")
    val = pd.DataFrame(rows); display(val)
    print("\nRead RECALL as the primary signal: the pipeline finds and links the annotated cells.")
    print("CHOTA / *F1 count every extra detection as a FP (lower bound); the host's sparse")
    print("node-count adjustment forgives most of those, so the leaderboard score is typically higher.")
except Exception as _e:
    print(f"Local validation skipped ({type(_e).__name__}: {str(_e)[:90]}).")
    print("This is expected on an offline Kaggle run (no geff/traccuracy) — the submission below is unaffected.")


## 7 · Generate `submission.csv` for all test movies

In [8]:

def run_submission(cfg=CFG):
    parts, stats = [], []
    for zp in sorted(TEST_DIR.glob("*.zarr")):
        ds = zp.name.replace(".zarr","")
        if not (zp/"0"/"zarr.json").exists():
            print("  (skip, no metadata)", ds); continue
        T, meta = avail_T(zp)                       # full on Kaggle; capped to present frames locally
        if T == 0:
            print("  (skip, no frames present)", ds); continue
        t0 = time.time()
        nodes, edges, st = track_movie(lambda t: load_volume(zp,t,meta), T, ds, cfg)
        st["sec"] = round(time.time()-t0,1); stats.append(st); parts += [nodes, edges]
        print(f"  {ds}: T={T} nodes={st['n_nodes']} edges={st['n_edges']} "
              f"cells/frame={st['cells_per_frame']:.1f} div={st['n_div']} ({st['sec']}s)")
    sub = pd.concat(parts, ignore_index=True); sub.index.name = "id"
    return sub, pd.DataFrame(stats)

submission, run_stats = run_submission(CFG)
submission.to_csv(OUT)
print("\nwrote", OUT, "rows:", len(submission))
display(run_stats)


In [9]:

# schema sanity-check against the expected columns
exp = ["dataset","row_type","node_id","t","z","y","x","source_id","target_id"]
assert list(submission.columns) == exp, submission.columns
nodes = submission[submission.row_type=="node"]; edges = submission[submission.row_type=="edge"]
assert (nodes[["source_id","target_id"]]==-1).all().all()
assert (edges[["node_id","t","z","y","x"]]==-1).all().all()
# every edge endpoint must be a real node id within its dataset
ok = True
for ds, g in submission.groupby("dataset"):
    ids = set(g[g.row_type=="node"].node_id); e = g[g.row_type=="edge"]
    if not (set(e.source_id) | set(e.target_id)).issubset(ids): ok = False; print("DANGLING EDGE in", ds)
print("schema OK:", ok, "| node rows:", len(nodes), "| edge rows:", len(edges), "| datasets:", submission.dataset.nunique())
display(submission.head(6)); display(submission.tail(4))


### Visual sanity-check — predicted tracks on a max-intensity projection

In [10]:

import matplotlib.pyplot as plt
# pick a test movie that has frames, overlay predicted nodes + short track segments on a Z-MIP
viz = None
for zp in sorted(TEST_DIR.glob("*.zarr")):
    T, meta = avail_T(zp)
    if T >= 3: viz = (zp, zp.name.replace(".zarr",""), T, meta); break
if viz:
    zp, ds, T, meta = viz
    g = submission[submission.dataset == ds]
    nd = g[g.row_type=="node"]; ed = g[g.row_type=="edge"]
    pos = {int(r.node_id): (r.x, r.y, int(r.t)) for r in nd.itertuples()}
    t_show = min(T-1, 30)
    vol = load_volume(zp, t_show, meta)
    fig, ax = plt.subplots(1, 2, figsize=(12, 5.2), constrained_layout=True)
    ax[0].imshow(vol.max(0), cmap="gray"); here = nd[nd.t==t_show]
    ax[0].scatter(here.x, here.y, s=14, c="#22d3ee", alpha=.8)
    ax[0].set_title(f"{ds} — detected cells at t={t_show} (n={len(here)})")
    ax[1].imshow(vol.max(0), cmap="gray")
    drawn = 0
    for s, tt in ed[["source_id","target_id"]].astype(int).values:
        if s in pos and tt in pos and drawn < 4000:
            ax[1].plot([pos[s][0], pos[tt][0]], [pos[s][1], pos[tt][1]], "-", color="#f59e0b", lw=.4, alpha=.5); drawn += 1
    ax[1].set_title(f"all predicted track links (Z-projected)")
    for a in ax: a.set_xlabel("x"); a.set_ylabel("y")
    plt.show()
    print(f"{ds}: {len(nd)} nodes, {len(ed)} edges over {T} frames — tracks follow the bright nuclei.")
else:
    print("no test movie with frames available locally for the overlay (runs fully on Kaggle)")


## 8 · How to climb from here

Validate **grouped by embryo** and change **one** thing per experiment:

* **Detection recall** is the main lever → lower `DOG_THR_PCT` (90 → 85) to keep more peaks. Do **not**
  cap detections by intensity: the GT nuclei are mid-pack in brightness, so a score-cap drops true cells.
* **Too few edges / broken tracks** → raise `MAX_LINK_DIST_UM` slightly (10 → 12) or keep `use_motion=True`.
* **Divisions** are **OFF by default** (GT mitoses are rare; false divisions tanked `div_f1` 1.0→0.0 in
  local tests). Enable `DETECT_DIVISIONS=True` only if validation on a division-rich movie shows a gain.
* **Isolated-node penalty** → keep `prune_isolated=True`; optionally `keep_strong_isolated_q≈0.8`.

**Next levels** (beyond this baseline): per-embryo count calibration, gap-closing across ≥2 frames
(LAP second pass), blob-scale (LoG) detection for varying nuclei sizes, and a learned 3D U-Net
nuclei detector trained on the dense-ish `6bba` GT. The detection step has the most headroom because
**edge recall ≈ node recall²**.
